# VI-Probe Generator — quickstart

Render the six variants of a classic visual illusion and inspect the sweep metadata.

Dependencies: `pip install -r requirements.txt` from the repo root. The first code cell
puts the repo root on `sys.path` so this notebook runs without any install.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "illusions").is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt

from illusions.registry import all_specs, get_illusion

[s.name for s in all_specs()][:10]

## The six variants

`original` (targets truly equal) vs `perturbed` (targets physically different, scaled by
`strength`), each with a matched `control` (illusion context removed) and a `with_guide`
version (guide markers reveal the true relation).

In [ ]:
VARIANTS = [
    ("original", dict(original=True)),
    ("original_with_guide", dict(original=True, visual_guide=True)),
    ("original_control", dict(control=True, original=True)),
    ("perturbed", dict(perturbed=True)),
    ("perturbed_with_guide", dict(perturbed=True, visual_guide=True)),
    ("perturbed_control", dict(control=True, perturbed=True)),
]

illusion = get_illusion("muller_lyer")()
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
for ax, (name, kwargs) in zip(axes.flat, VARIANTS):
    illusion.set_variation(**kwargs)
    ax.imshow(illusion.generate(strength=1.3, save=False))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()

## Strength sweep

Strength is multiplicative around 1.0 — the published perturbed sweep covers 0.5–1.5.

In [ ]:
illusion.set_variation(perturbed=True)
fig, axes = plt.subplots(1, 4, figsize=(16, 2.5))
for ax, s in zip(axes, [0.6, 0.9, 1.1, 1.4]):
    ax.imshow(illusion.generate(strength=s, save=False))
    ax.set_title(f"strength={s}")
    ax.axis("off")
plt.tight_layout()

## Sweep metadata (no images)

The same machinery that produced the published dataset:

In [ ]:
import pandas as pd

from sweep import load_config, sweep_class

cfg = load_config(repo_root / "configs" / "sweeps" / "size.yaml")
records = sweep_class(cfg, "MullerLyerIllusion", "unused", generate_images=False)
pd.DataFrame(records).head()

To run a full sweep (images + metadata):
`python main.py --sweep configs/sweeps/size.yaml -o output/size_sweep`
(see `docs/sweep.md`).